# 02 — Train Random Forest trên dữ liệu Parquet thật (DEV)

Notebook này được tạo để **chạy trong repo `MachineLearningRandomForest/notebooks/`**.

**DEV v2 (sửa lỗi schema):** dữ liệu Parquet hiện thiếu `characteristics` và `body_type_name` so với `configs/ds04_schema.json`. Notebook cho phép *tạm thời loại khỏi danh sách feature khi train DEV* **chỉ hai tên cột này**, có kiểm tra train/val nhất quán và ghi rõ khác biệt trong artifacts. Điều này **không thay đổi data contract hoặc dữ liệu**; cần TV1/TV2 xác nhận trước kết quả cuối. Nếu thiếu bất kỳ cột khác, notebook sẽ dừng. Mọi thay đổi phải thực hiện **ở local**, commit/push GitHub rồi mới `git pull` và chạy trên Kaggle.

- Tận dụng `mlrf.training.build_model('custom_random_forest', ...)` của nhóm (không thay bằng `sklearn.RandomForestRegressor`).
- **Không chia tập lại**: chỉ fit trên `train.parquet`, đánh giá trên `val.parquet`.
- **Không đọc, fit, hoặc tính metrics trên `test.parquet`** trong giai đoạn DEV; chỉ kiểm tra tên file test tồn tại.
- Tiền xử lý trong notebook là **bản ghép nối tạm thời (provisional)**, giống hướng tiếp cận smoke test. Chưa thay thế quy trình tiền xử lý chính thức của TV2.
- `dataset.approved` được giữ **False** tới khi nhóm xác nhận data contract / quality audit; metrics ở đây **không phải kết quả cuối của dự án**.

Chạy lần đầu với **20.000 dòng train** để kiểm tra độ ổn định và RAM; sau khi ổn định, đổi `MAX_TRAIN_ROWS = None` để train toàn bộ tập train. Đọc dữ liệu mặc định từ Kaggle `ncd426/machinelearning`, hoặc đặt biến môi trường `MLRF_DATA_DIR` khi chạy local.

## 1. Cấu hình và import từ source gốc của nhóm

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Dùng đúng source trong repo đã clone; hỗ trợ chạy local từ thư mục gốc hoặc notebooks/.
REPO = Path('/kaggle/working/MachineLearningRandomForest')
if not (REPO / 'src' / 'mlrf').is_dir():
    REPO = Path.cwd().resolve()
    if REPO.name == 'notebooks':
        REPO = REPO.parent
if not (REPO / 'src' / 'mlrf').is_dir():
    raise FileNotFoundError('Không tìm thấy repo có src/mlrf. Hãy chạy notebook từ root hoặc notebooks/ của repo.')

os.chdir(REPO)  # load_project_config() / load_ds04_contract() có thể dùng đường dẫn tương đối.
sys.path.insert(0, str(REPO / 'src'))

from mlrf.config import load_project_config
from mlrf.data.contract import load_ds04_contract
from mlrf.evaluation import regression_metrics
from mlrf.ops import environment_info, save_run
from mlrf.testing import assert_target_absent
from mlrf.training import build_model

CONF = load_project_config()
CONTRACT = load_ds04_contract()
SEED = int(CONF['split']['random_seed'])
TARGET = CONF['dataset']['target']

DATA_DIR = Path(os.environ.get('MLRF_DATA_DIR', '/kaggle/input/datasets/ncd426/machinelearning'))
MAX_TRAIN_ROWS = 20_000       # Đặt None sau khi kiểm tra thành công để train TOÀN BỘ tập train.
MAX_VAL_ROWS = None          # Giữ None để đánh giá trên toàn bộ val.parquet.
VAL_PRED_BATCH = 2_048       # Dự đoán VAL thành nhiều batch để tránh one-hot chiếm quá nhiều RAM.
MAX_DENSE_MB = 1_200        # Giới hạn sơ bộ RAM ma trận one-hot dạng dense (chưa gồm overhead).
RUN_ID = 'real_dev_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
MODEL_PARAMS = {
    'n_estimators': 12,
    'max_depth': 7,
    'max_features': 0.7,
    'bootstrap': True,
    'oob_score': True,
    'random_state': SEED,
}

print('Repo:', REPO)
print('Data:', DATA_DIR)
print('Target:', TARGET, '| Seed:', SEED)
print('Python/libraries:', environment_info())
print('Run ID:', RUN_ID)
print('DEV run: not final, dataset approval not claimed.')

## 2. Đọc đúng các split đã có, không chia lại

In [ ]:
required = ['train.parquet', 'val.parquet', 'test.parquet']
for name in required:
    path = DATA_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {path}. Hãy gắn Kaggle Dataset hoặc cấu hình MLRF_DATA_DIR.')

train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'val.parquet')

print('Train shape:', train_df.shape)
print('Val shape:', val_df.shape)
print('Test file:', DATA_DIR / 'test.parquet', '(chưa đọc nội dung test)')

# Hiện các báo cáo đi kèm để người phụ trách xác nhận trước khi tuyên bố kết quả cuối.
for name in ['cleaning_report.json', 'preprocessing_check.json', 'split_report.json']:
    p = DATA_DIR / name
    if p.exists():
        try:
            report = json.loads(p.read_text(encoding='utf-8'))
            keys = sorted(report.keys()) if isinstance(report, dict) else []
            print(f'{name}: tồn tại; keys = {keys[:30]}')
        except (ValueError, UnicodeError) as exc:
            print(f'{name}: không đọc được JSON: {exc}')

## 3. Kiểm tra schema, tạo X/y từ contract của dự án

In [ ]:
# Data contract vẫn là nguồn tham chiếu; không sửa dữ liệu, không thêm cột giả.
contract_numeric = [c for c in CONTRACT['numeric_columns'] if c != TARGET]
contract_categorical = [c for c in CONTRACT['categorical_columns'] if c != TARGET]
expected_features = contract_numeric + contract_categorical

if not expected_features:
    raise ValueError('Data contract không có feature để train.')
if len(expected_features) != len(set(expected_features)):
    raise ValueError('Data contract có feature bị trùng giữa numeric/categorical.')

# DEV waiver: chỉ hai cột này đang được báo thiếu trong dữ liệu thật.
# TODO: TV1/TV2 xác nhận schema chính thức trước khi xóa waiver.
DEV_MISSING_FEATURE_ALLOWLIST = {'characteristics', 'body_type_name'}
missing_by_split = {
    label: sorted(set(expected_features) - set(df.columns))
    for label, df in [('train', train_df), ('val', val_df)]
}
print('Data contract features:', len(expected_features))
print('Train columns:', train_df.columns.tolist())
print('Validation columns:', val_df.columns.tolist())
print('Missing features by split:', missing_by_split)

missing_train = set(missing_by_split['train'])
missing_val = set(missing_by_split['val'])
if missing_train != missing_val:
    raise ValueError('Train/val thiếu các feature KHÁC nhau; phải đối chiếu pipeline chia dữ liệu trước khi train.')
unexpected_missing = missing_train - DEV_MISSING_FEATURE_ALLOWLIST
if unexpected_missing:
    raise ValueError(f'Thiếu cột ngoài DEV allowlist: {sorted(unexpected_missing)}. Không tự bỏ cột.')

missing_contract_features = sorted(missing_train)
if missing_contract_features:
    print('WARNING DEV: Bỏ qua tạm thời feature thiếu trong Parquet:', missing_contract_features)
    print('WARNING DEV: KHÔNG được dùng metrics này làm kết quả cuối trước khi TV1/TV2 nghiệm thu.')

numeric = [c for c in contract_numeric if c not in missing_train]
categorical = [c for c in contract_categorical if c not in missing_train]
features = numeric + categorical
if not features:
    raise ValueError('Không còn feature để train sau khi đối chiếu schema.')

for label, df in [('train', train_df), ('val', val_df)]:
    if TARGET not in df.columns:
        raise ValueError(f'{label}.parquet thiếu cột mục tiêu {TARGET!r}.')
    if not pd.api.types.is_numeric_dtype(df[TARGET]):
        raise TypeError(f'{label}.{TARGET} chưa có kiểu số. Cần xử lý ở pipeline dữ liệu trước.')
    if df[TARGET].isna().any() or not np.isfinite(df[TARGET].to_numpy(dtype=float)).all():
        raise ValueError(f'{label}.{TARGET} có null/inf. Cần xử lý dữ liệu trước khi train.')
    wrong_numeric = [c for c in numeric if not pd.api.types.is_numeric_dtype(df[c])]
    if wrong_numeric:
        raise TypeError(f'{label}: cột numeric sai kiểu: {wrong_numeric}')

# Lấy mẫu chỉ TRONG tập train; không gộp hay chia lại các tập.
train_use = (train_df.sample(n=MAX_TRAIN_ROWS, random_state=SEED)
             if MAX_TRAIN_ROWS is not None and len(train_df) > MAX_TRAIN_ROWS
             else train_df)
val_use = (val_df.sample(n=MAX_VAL_ROWS, random_state=SEED)
           if MAX_VAL_ROWS is not None and len(val_df) > MAX_VAL_ROWS
           else val_df)

def make_x(df):
    x = df[features].copy()  # Không đưa cột target hay những cột ngoài contract vào model.
    for col in categorical:
        # Chuẩn hóa kiểu để SimpleImputer/OneHotEncoder không gặp hỗn hợp string + pd.NA.
        x[col] = x[col].astype('string').fillna('__MISSING__').astype(object)
    return x

X_train = make_x(train_use)
y_train = train_use[TARGET].copy()
X_val = make_x(val_use)
y_val = val_use[TARGET].copy()

assert_target_absent(X_train, target=TARGET)
assert_target_absent(X_val, target=TARGET)
print('Train used:', len(X_train), 'of', len(train_df))
print('Val used:', len(X_val), 'of', len(val_df))
print('Numeric:', numeric)
print('Categorical:', categorical)
print('Extra columns excluded:', sorted(set(train_df.columns) - set(features) - {TARGET}))

## 4. Giới hạn an toàn RAM và tạo model từ `mlrf`

`01_training_smoke` đã dùng `ColumnTransformer`, `SimpleImputer`, `OneHotEncoder` cùng custom Random Forest. Notebook này dùng cùng hướng ghép nối, nhưng **ép one-hot về dense** nhằm tránh việc custom estimator không hỗ trợ sparse. Vì one-hot dense có thể rất lớn, notebook sẽ **chặn trước khi fit** nếu ước lượng bộ nhớ vượt ngưỡng. Đây là ước lượng tối thiểu, không phải dự báo tổng RAM.

In [ ]:
cardinality = {c: int(X_train[c].nunique(dropna=False)) for c in categorical}
feature_dim_estimate = len(numeric) + sum(cardinality.values())
dense_mb_estimate = len(X_train) * feature_dim_estimate * 8 / (1024 ** 2)
print('Category cardinality:', cardinality)
print('Estimated one-hot feature count:', feature_dim_estimate)
print(f'Estimated dense TRAIN matrix only: {dense_mb_estimate:,.1f} MiB')

if dense_mb_estimate > MAX_DENSE_MB:
    raise MemoryError(
        f'Ma trận one-hot ước tính {dense_mb_estimate:,.0f} MiB > {MAX_DENSE_MB} MiB. '
        'Giảm MAX_TRAIN_ROWS hoặc phối hợp TV2 để tối ưu encode trước khi train full.'
    )

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric),
    ('cat', Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical),
])

model = Pipeline([
    ('preprocess', pre),
    ('regressor', build_model('custom_random_forest', MODEL_PARAMS)),
])
print('Model type:', type(model.named_steps['regressor']).__name__)
print('Model source:', model.named_steps['regressor'].__class__.__module__)

## 5. Fit trên TRAIN và đánh giá DEV trên VAL (không đụng đến TEST)

In [ ]:
start = time.perf_counter()
model.fit(X_train, y_train)  # Preprocessing chỉ được fit bằng TRAIN.
fit_seconds = time.perf_counter() - start

# Dự đoán theo batch: không tạo một ma trận one-hot lớn cho 64k dòng validation.
val_preds = []
for start in range(0, len(X_val), VAL_PRED_BATCH):
    batch = X_val.iloc[start:start + VAL_PRED_BATCH]
    val_preds.append(np.asarray(model.predict(batch)))
pred_val = np.concatenate(val_preds)
metrics = regression_metrics(y_val, pred_val)

print(f'Fit completed in {fit_seconds:,.1f} seconds')
print('DEV validation metrics (NOT FINAL):', metrics)
print('Custom RF OOB:', getattr(model.named_steps['regressor'], 'oob_score_', 'not available'))
print('TEST HAS NOT BEEN USED.')

## 6. Lưu run, dự đoán validation và model vào output

In [ ]:
out_root = (Path('/kaggle/working/mlrf_runs')
            if Path('/kaggle/working').exists()
            else REPO / 'outputs' / 'mlrf_runs')
out_root.mkdir(parents=True, exist_ok=True)

commit = subprocess.run(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=False,
)
git_commit = commit.stdout.strip() if commit.returncode == 0 else None

dataset_identity = {
    'source': 'kaggle:ncd426/machinelearning',
    'path': str(DATA_DIR),
    'approved': False,
    'rows': len(train_use),
    'train_rows_total': len(train_df),
    'train_rows_used': len(train_use),
    'val_rows_used': len(val_use),
    'test_used': False,
    'missing_contract_features': missing_contract_features,
    'contract_match': len(missing_contract_features) == 0,
}

run_config = {
    'model': 'custom_random_forest',
    **MODEL_PARAMS,
    'seed': SEED,
    'split_status': 'pre-split data; validation only, no test used',
    'preprocessing_status': 'provisional adapter, NOT TV2 final',
    'schema_status': 'DEV waiver: missing contract features' if missing_contract_features else 'feature names match contract',
    'missing_contract_features': missing_contract_features,
    'training_seconds': fit_seconds,
    'full_train': len(train_use) == len(train_df),
}

run_path = Path(save_run(
    out_root,
    RUN_ID,
    config=run_config,
    metrics=metrics,
    dataset=dataset_identity,
    git_commit=git_commit,
    predictions={
        'row_id': np.arange(len(val_use), dtype=np.int64),
        'actual': y_val.to_numpy(),
        'predicted': np.asarray(pred_val),
    },
))

model_path = run_path / 'model.joblib'
joblib.dump(model, model_path, compress=3)
print('Artifacts:', run_path)
print('Model:', model_path)
print('Git commit:', git_commit)
print('DEV ONLY: dữ liệu/preprocessing chưa được xác nhận approved.')

## 7. Chuyển sang full train và lưu kết quả

Nếu chạy **20.000 dòng** thành công và không vượt RAM:

**Chú ý:** Ở bản DEV v2, việc bỏ qua `characteristics` và `body_type_name` chỉ là ngoại lệ tạm. Trước khi train bản cuối, TV1/TV2 phải xác nhận dữ liệu có chủ đích loại chúng hoặc cập nhật schema tương ứng. **Không đánh giá kết quả này là final.**


1. Tại **máy local**, sửa cấu hình trong cell 1: `MAX_TRAIN_ROWS = None` để huấn luyện toàn bộ `train.parquet` (302.471 dòng theo lần kiểm tra trước).
2. Commit/push file `notebooks/02_train_real.ipynb` lên GitHub; **không sửa notebook trực tiếp trên Kaggle**.
3. Trên Kaggle, cập nhật source và chạy lại notebook. Nếu gặp `MemoryError`, cần trao đổi với TV2 để tối ưu mã hóa biến phân loại hoặc giới hạn tài nguyên; **không tự bỏ cột hoặc thay đổi thuật toán của nhóm**.
4. Dữ liệu `test.parquet` tiếp tục giữ nguyên tới khi chốt model, hoàn tất kiểm tra data leakage và được người phụ trách phê duyệt.
5. Sau lần chạy thành công, chọn **Save Version → Save & Run All** trên Kaggle để giữ output. Kết quả nằm trong `/kaggle/working/mlrf_runs/<RUN_ID>/`.

**Ghi chú:** Đoạn mã này dựa vào API quan sát được trong notebook smoke của repo (`build_model`, `load_ds04_contract`, `regression_metrics`, `save_run`). Chưa xác minh trực tiếp được nội dung source Github, vì vậy nếu module thay đổi chữ ký hàm thì cần đối chiếu lại với code local.